# Produce data

- Mô phỏng quá trình generate và đẩy dữ liệu vào db của một cty
- Mỗi giây sẽ chèn tầm 17_000 record vào db

In [2]:
import sys
sys.path.append('/home/longnb32/FPT/ops/ml-ops/src')

import time
import numpy as np
import pandas as pd
from utils.db_connection import PostgresConnection
import s3fs
import os

- Thông tin cột, kiểu dữ liệu của cột, mapping để đổi tên cột

In [3]:
LIST_COLS = [
    "vendor_id", "pickup_datetime", "dropoff_datetime", "passenger_count",
    "trip_distance", "rate_code_id", "store_and_fwd_flag", "pickup_longitude",
    "pickup_latitude", "dropoff_longitude", "dropoff_latitude", "pu_location_id",
    "do_location_id", "payment_type", "fare_amount", "extra", "mta_tax",
    "surcharge", "improvement_surcharge", "tip_amount", "tolls_amount",
    "total_amount", "congestion_surcharge", "airport_fee", "cbd_congestion_fee",
]

LIST_DTYPES = {
    "vendor_id":             "object",
    "pickup_datetime":       "datetime64[ns]",
    "dropoff_datetime":      "datetime64[ns]",
    "passenger_count":       "float64",
    "trip_distance":         "float64",
    "rate_code_id":          "float64",
    "store_and_fwd_flag":    "object",
    "pickup_longitude":      "float64",
    "pickup_latitude":       "float64",
    "dropoff_longitude":     "float64",
    "dropoff_latitude":      "float64",
    "pu_location_id":        "object",
    "do_location_id":        "object",
    "payment_type":          "object",
    "fare_amount":           "float64",
    "extra":                 "float64",
    "mta_tax":               "float64",
    "surcharge":             "float64",
    "improvement_surcharge": "float64",
    "tip_amount":            "float64",
    "tolls_amount":          "float64",
    "total_amount":          "float64",
    "congestion_surcharge":  "float64",
    "airport_fee":           "float64",
    "cbd_congestion_fee":    "float64",
}

map1 = {
    "columns": 'VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee',
    'mapping': {
        "VendorID": "vendor_id",
        "tpep_pickup_datetime": "pickup_datetime",
        "tpep_dropoff_datetime": "dropoff_datetime",
        "RatecodeID": "rate_code_id",
        "PULocationID": "pu_location_id",
        "DOLocationID": "do_location_id",
        "payment_type": "payment_type",
        "fare_amount": "fare_amount",
    }
}

map2 = {
    'columns':  'VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee',
    'mapping': {
        "VendorID": "vendor_id",
        "tpep_pickup_datetime": "pickup_datetime",
        "tpep_dropoff_datetime": "dropoff_datetime",
        "RatecodeID": "rate_code_id",
        "PULocationID": "pu_location_id",
        "DOLocationID": "do_location_id",
        "Airport_fee": "airport_fee",
    }
}

map3 = {
    'columns': 'vendor_name|Trip_Pickup_DateTime|Trip_Dropoff_DateTime|Passenger_Count|Trip_Distance|Start_Lon|Start_Lat|Rate_Code|store_and_forward|End_Lon|End_Lat|Payment_Type|Fare_Amt|surcharge|mta_tax|Tip_Amt|Tolls_Amt|Total_Amt',
    'mapping': {
        "vendor_name": "vendor_id",
        "Trip_Pickup_DateTime": "pickup_datetime",
        "Trip_Dropoff_DateTime": "dropoff_datetime",
        "Passenger_Count": "passenger_count",
        "Trip_Distance": "trip_distance",
        "Start_Lon": "pickup_longitude",
        "Start_Lat": "pickup_latitude",
        "Rate_Code": "rate_code_id",
        "store_and_forward": "store_and_fwd_flag",
        "End_Lon": "dropoff_longitude",
        "End_Lat": "dropoff_latitude",
        "Payment_Type": "payment_type",
        "Fare_Amt": "fare_amount",
        "Tip_Amt": "tip_amount",
        "Tolls_Amt": "tolls_amount",
        "Total_Amt": "total_amount",
    }
}

map4 = {
    'columns': 'VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee',
    'mapping': {
        "VendorID": "vendor_id",
        "tpep_pickup_datetime": "pickup_datetime",
        "tpep_dropoff_datetime": "dropoff_datetime",
        "RatecodeID": "rate_code_id",
        "PULocationID": "pu_location_id",
        "DOLocationID": "do_location_id",
        "Airport_fee": "airport_fee",
    }
}

map5 = {
    'columns':  'vendor_id|pickup_datetime|dropoff_datetime|passenger_count|trip_distance|pickup_longitude|pickup_latitude|rate_code|store_and_fwd_flag|dropoff_longitude|dropoff_latitude|payment_type|fare_amount|surcharge|mta_tax|tip_amount|tolls_amount|total_amount',
    'mapping': {
        "rate_code": "rate_code_id",
    }
}

map6 = {
    'columns':  'vendorID|tpepPickupDateTime|tpepDropoffDateTime|passengerCount|tripDistance|puLocationId|doLocationId|startLon|startLat|endLon|endLat|rateCodeId|storeAndFwdFlag|paymentType|fareAmount|extra|mtaTax|improvementSurcharge|tipAmount|tollsAmount|totalAmount',
    'mapping': {
        "vendorID": "vendor_id",
        "tpepPickupDateTime": "pickup_datetime",
        "tpepDropoffDateTime": "dropoff_datetime",
        "passengerCount": "passenger_count",
        "tripDistance": "trip_distance",
        "puLocationId": "pu_location_id",
        "doLocationId": "do_location_id",
        "startLon": "pickup_longitude",
        "startLat": "pickup_latitude",
        "endLon": "dropoff_longitude",
        "endLat": "dropoff_latitude",
        "rateCodeId": "rate_code_id",
        "storeAndFwdFlag": "store_and_fwd_flag",
        "paymentType": "payment_type",
        "fareAmount": "fare_amount",
        "mtaTax": "mta_tax",
        "improvementSurcharge": "improvement_surcharge",
        "tipAmount": "tip_amount",
        "tollsAmount": "tolls_amount",
        "totalAmount": "total_amount",
    }
}

SCHEMA_MAPPINGS = [map1, map2, map3, map4, map5, map6]

- Tạo bảng

In [4]:
def create_table(conn):
    """Tạo bảng nyc_taxi nếu chưa tồn tại."""
    query = """
    CREATE TABLE IF NOT EXISTS public.nyc_taxi (
        vendor_id               TEXT,
        pickup_datetime         TIMESTAMP,
        dropoff_datetime        TIMESTAMP,
        passenger_count         FLOAT,
        trip_distance           FLOAT,
        rate_code_id            FLOAT,
        store_and_fwd_flag      TEXT,
        pickup_longitude        FLOAT,
        pickup_latitude         FLOAT,
        dropoff_longitude       FLOAT,
        dropoff_latitude        FLOAT,
        pu_location_id          TEXT,
        do_location_id          TEXT,
        payment_type            TEXT,
        fare_amount             FLOAT,
        extra                   FLOAT,
        mta_tax                 FLOAT,
        surcharge               FLOAT,
        improvement_surcharge   FLOAT,
        tip_amount              FLOAT,
        tolls_amount            FLOAT,
        total_amount            FLOAT,
        congestion_surcharge    FLOAT,
        airport_fee             FLOAT,
        cbd_congestion_fee      FLOAT
    );
    """
    with conn.cursor() as cur:
        cur.execute(query)
    conn.commit()
    print("Table 'nyc_taxi' created successfully.")

- Transform

In [5]:
def _resolve_mapping(df_columns: str) -> dict:
    """Tìm schema mapping khớp với columns của file."""
    for schema in SCHEMA_MAPPINGS:
        if schema["columns"] == df_columns:
            return schema["mapping"]
    raise ValueError(f"No mapping found for {df_columns}")

def transform_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    1. Đổi tên cột theo mapping tương ứng.
    2. Thêm cột canonical còn thiếu (fill NaN).
    3. Đổi kiểu dữ liệu về canonical dtype.
    4. Thay NaN → None để insert NULL vào Postgres.
    """
    # 1. Resolve & rename
    mapping = _resolve_mapping('|'.join(df.columns))
    df.rename(columns=mapping, inplace=True)

    # 2. Thêm cột còn thiếu
    for col in LIST_COLS:
        if col not in df.columns:
            df[col] = np.nan

    # 3. Cast dtype
    for col, dtype in LIST_DTYPES.items():
        if dtype == "datetime64[ns]":
            df[col] = pd.to_datetime(df[col], errors="coerce")
        elif dtype == "float64":
            df[col] = pd.to_numeric(df[col], errors="coerce")
        else:  # object
            df[col] = df[col].astype("object")

    # 4. NaN → None
    return df.astype(object).where(df.notna(), None)[LIST_COLS]

In [6]:
# fs = s3fs.S3FileSystem()
df_original = pd.read_parquet('../../data/test/parquet/yellow_tripdata_2010-01.parquet')
df_original.head()

,vendor_id,pickup_datetime,dropoff_datetime,passenger_count,trip_distance,pickup_longitude,pickup_latitude,rate_code,store_and_fwd_flag,dropoff_longitude,dropoff_latitude,payment_type,fare_amount,surcharge,mta_tax,tip_amount,tolls_amount,total_amount
0,VTS,2010-01-26 07:41:00,2010-01-26 07:45:00,1,0.75,-73.956778,40.767750,1,None,-73.965957,40.765232,CAS,4.5,0.0,0.5,0.00,0.0,5.00
1,DDS,2010-01-30 23:31:00,2010-01-30 23:46:12,1,5.90,-73.996118,40.763932,1,None,-73.981512,40.741193,CAS,15.3,0.5,0.5,0.00,0.0,16.30
2,DDS,2010-01-18 20:22:20,2010-01-18 20:38:12,1,4.00,-73.979673,40.783790,1,None,-73.917852,40.878560,CAS,11.7,0.5,0.5,0.00,0.0,12.70
3,VTS,2010-01-09 01:18:00,2010-01-09 01:35:00,2,4.70,-73.977922,40.763997,1,None,-73.923908,40.759725,CAS,13.3,0.5,0.5,0.00,0.0,14.30
4,CMT,2010-01-18 19:10:14,2010-01-18 19:17:07,1,0.60,-73.990924,40.734682,1,0,-73.995511,40.739088,Cre,5.3,0.0,0.5,0.87,0.0,6.67


In [7]:
df_original.describe()

,passenger_count,trip_distance,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,fare_amount,surcharge,mta_tax,tip_amount,tolls_amount,total_amount
count,1.486378e+07,1.486378e+07,1.486378e+07,1.486378e+07,1.486378e+07,1.486378e+07,1.486378e+07,1.486378e+07,1.486378e+07,1.486378e+07,1.486378e+07,1.486378e+07
mean,1.684686e+00,2.628267e+00,-7.263310e+01,4.001195e+01,-7.265785e+01,4.002671e+01,9.427919e+00,3.196215e-01,4.706081e-01,6.714118e-01,1.470634e-01,1.106182e+01
std,1.260632e+00,3.018446e+00,9.901503e+00,5.628441e+00,9.823565e+00,5.440934e+00,7.379477e+00,3.663894e-01,1.937933e-01,1.464261e+00,8.585346e-01,8.594386e+00
min,0.000000e+00,0.000000e+00,-7.832167e+02,-3.538407e+03,-2.069479e+03,-2.269701e+00,2.500000e+00,0.000000e+00,-1.000000e+00,0.000000e+00,0.000000e+00,2.500000e+00
25%,1.000000e+00,1.000000e+00,-7.399174e+01,4.073574e+01,-7.399124e+01,4.073492e+01,5.300000e+00,0.000000e+00,5.000000e-01,0.000000e+00,0.000000e+00,6.440000e+00
50%,1.000000e+00,1.680000e+00,-7.398161e+01,4.075345e+01,-7.398010e+01,4.075376e+01,7.300000e+00,0.000000e+00,5.000000e-01,0.000000e+00,0.000000e+00,8.600000e+00
75%,2.000000e+00,2.960000e+00,-7.396747e+01,4.076774e+01,-7.396432e+01,4.076871e+01,1.050000e+01,5.000000e-01,5.000000e-01,1.000000e+00,0.000000e+00,1.220000e+01
max,2.080000e+02,5.000000e+01,8.434618e+01,1.541867e+03,4.276011e+01,1.541867e+03,2.000000e+02,1.050000e+01,5.000000e-01,1.000000e+02,2.000000e+01,2.360000e+02


In [8]:
df = transform_data(df_original)
df.head()

,vendor_id,pickup_datetime,dropoff_datetime,passenger_count,trip_distance,rate_code_id,store_and_fwd_flag,pickup_longitude,pickup_latitude,dropoff_longitude,...,extra,mta_tax,surcharge,improvement_surcharge,tip_amount,tolls_amount,total_amount,congestion_surcharge,airport_fee,cbd_congestion_fee
0,VTS,2010-01-26 07:41:00,2010-01-26 07:45:00,1,0.75,1,None,-73.956778,40.76775,-73.965957,...,None,0.5,0.0,None,0.0,0.0,5.0,None,None,None
1,DDS,2010-01-30 23:31:00,2010-01-30 23:46:12,1,5.9,1,None,-73.996118,40.763932,-73.981512,...,None,0.5,0.5,None,0.0,0.0,16.3,None,None,None
2,DDS,2010-01-18 20:22:20,2010-01-18 20:38:12,1,4.0,1,None,-73.979673,40.78379,-73.917852,...,None,0.5,0.5,None,0.0,0.0,12.7,None,None,None
3,VTS,2010-01-09 01:18:00,2010-01-09 01:35:00,2,4.7,1,None,-73.977922,40.763997,-73.923908,...,None,0.5,0.5,None,0.0,0.0,14.3,None,None,None
4,CMT,2010-01-18 19:10:14,2010-01-18 19:17:07,1,0.6,1,0,-73.990924,40.734682,-73.995511,...,None,0.5,0.0,None,0.87,0.0,6.67,None,None,None


In [9]:
df.describe()

/tmp/ipykernel_2899741/3627053830.py:1: FutureWarning: The behavior of value_counts with object-dtype is deprecated. In a future version, this will *not* perform dtype inference on the resulting index. To retain the old behavior, use `result.index = result.index.infer_objects()`
  df.describe()
/tmp/ipykernel_2899741/3627053830.py:1: FutureWarning: The behavior of value_counts with object-dtype is deprecated. In a future version, this will *not* perform dtype inference on the resulting index. To retain the old behavior, use `result.index = result.index.infer_objects()`
  df.describe()


,vendor_id,pickup_datetime,dropoff_datetime,passenger_count,trip_distance,rate_code_id,store_and_fwd_flag,pickup_longitude,pickup_latitude,dropoff_longitude,...,extra,mta_tax,surcharge,improvement_surcharge,tip_amount,tolls_amount,total_amount,congestion_surcharge,airport_fee,cbd_congestion_fee
count,14863778,14863778,14863778,14863778,14863778.0,14863778,6571159,14863778.0,14863778.0,14863778.0,...,0,14863778.0,14863778.0,0,14863778.0,14863778.0,14863778.0,0,0,0
unique,3,2292957,2295565,8,3764.0,1,4,212644.0,211355.0,267309.0,...,0,3.0,27.0,0,2445.0,569.0,9386.0,0,0,0
top,VTS,2010-01-29 19:31:00,2010-01-29 19:47:00,1,0.9,1,0,0.0,0.0,0.0,...,NaN,0.5,0.0,NaN,0.0,0.0,5.4,NaN,NaN,NaN
freq,7448983,389,400,10109250,361601.0,14863778,6446823,268331.0,268241.0,263146.0,...,NaN,14460194.0,7603158.0,NaN,10214206.0,14412735.0,356319.0,NaN,NaN,NaN


In [5]:
def insert_parquet(conn, filepath: str, n: int = 1, batch_size: int = 17_000, interval: float = 1.0):
    """Đọc file parquet, lấy n sample cách đều, rồi insert theo batch."""
    df = transform_data(pd.read_parquet(filepath))

    # Lấy n sample cách đều
    # Tôi muốn xử lý xong df rồi mới sample để check nếu nó có lỗi gì
    indices = np.linspace(0, len(df) - 1, num=n, dtype=int)
    df = df.iloc[indices].reset_index(drop=True)

    placeholders = ', '.join(['%s'] * len(LIST_COLS))
    insert_query = f"INSERT INTO nyc_taxi ({', '.join(LIST_COLS)}) VALUES ({placeholders})"

    total_rows = len(df)
    inserted = 0

    with conn.cursor() as cur:
        while inserted < total_rows:
            batch = df.iloc[inserted:inserted + batch_size]
            values = [tuple(row) for row in batch.itertuples(index=False, name=None)]
            cur.executemany(insert_query, values)
            conn.commit()
            inserted += len(batch)
            print(f"Inserted {inserted}/{total_rows} rows")
            if inserted < total_rows:
                time.sleep(interval)

    print(f"Done! Inserted {total_rows} rows from {filepath}")

In [14]:
df = transform_data(pd.read_parquet('s3://flezi-agent/nyc-taxi/yellow_tripdata_2009-01.parquet'))
df.head()

,vendor_id,pickup_datetime,dropoff_datetime,passenger_count,trip_distance,rate_code_id,store_and_fwd_flag,pickup_longitude,pickup_latitude,dropoff_longitude,...,extra,mta_tax,surcharge,improvement_surcharge,tip_amount,tolls_amount,total_amount,congestion_surcharge,airport_fee,cbd_congestion_fee
0,VTS,2009-01-04 02:52:00,2009-01-04 03:02:00,1,2.63,NaN,NaN,-73.991957,40.721567,-73.993803,...,NaN,NaN,0.5,NaN,0.00,0.0,9.40,NaN,NaN,NaN
1,VTS,2009-01-04 03:31:00,2009-01-04 03:38:00,3,4.55,NaN,NaN,-73.982102,40.736290,-73.955850,...,NaN,NaN,0.5,NaN,2.00,0.0,14.60,NaN,NaN,NaN
2,VTS,2009-01-03 15:43:00,2009-01-03 15:57:00,5,10.35,NaN,NaN,-74.002587,40.739748,-73.869983,...,NaN,NaN,0.0,NaN,4.74,0.0,28.44,NaN,NaN,NaN
3,DDS,2009-01-01 20:52:58,2009-01-01 21:14:00,1,5.00,NaN,NaN,-73.974267,40.790955,-73.996558,...,NaN,NaN,0.5,NaN,3.05,0.0,18.45,NaN,NaN,NaN
4,DDS,2009-01-24 16:18:23,2009-01-24 16:24:56,1,0.40,NaN,NaN,-74.001580,40.719382,-74.008378,...,NaN,NaN,0.0,NaN,0.00,0.0,3.70,NaN,NaN,NaN


In [12]:
df[df.congestion_surcharge.isna()].congestion_surcharge = None
df.head()

,vendor_id,pickup_datetime,dropoff_datetime,passenger_count,trip_distance,rate_code_id,store_and_fwd_flag,pickup_longitude,pickup_latitude,dropoff_longitude,...,extra,mta_tax,surcharge,improvement_surcharge,tip_amount,tolls_amount,total_amount,congestion_surcharge,airport_fee,cbd_congestion_fee
0,VTS,2009-01-04 02:52:00,2009-01-04 03:02:00,1,2.63,NaN,None,-73.991957,40.721567,-73.993803,...,NaN,NaN,0.5,NaN,0.00,0.0,9.40,NaN,NaN,NaN
1,VTS,2009-01-04 03:31:00,2009-01-04 03:38:00,3,4.55,NaN,None,-73.982102,40.736290,-73.955850,...,NaN,NaN,0.5,NaN,2.00,0.0,14.60,NaN,NaN,NaN
2,VTS,2009-01-03 15:43:00,2009-01-03 15:57:00,5,10.35,NaN,None,-74.002587,40.739748,-73.869983,...,NaN,NaN,0.0,NaN,4.74,0.0,28.44,NaN,NaN,NaN
3,DDS,2009-01-01 20:52:58,2009-01-01 21:14:00,1,5.00,NaN,None,-73.974267,40.790955,-73.996558,...,NaN,NaN,0.5,NaN,3.05,0.0,18.45,NaN,NaN,NaN
4,DDS,2009-01-24 16:18:23,2009-01-24 16:24:56,1,0.40,NaN,None,-74.001580,40.719382,-74.008378,...,NaN,NaN,0.0,NaN,0.00,0.0,3.70,NaN,NaN,NaN


In [ ]:
pg = PostgresConnection(port=5432)
with pg as conn:
    create_table(conn)
    insert_parquet(conn, filepath='s3://flezi-data/nyc-taxi/yellow_tripdata_2009-01.parquet')

Table 'nyc_taxi' created successfully.
Inserted 1/1 rows
Done! Inserted 1 rows from s3://flezi-agent/nyc-taxi/yellow_tripdata_2009-01.parquet


In [ ]:
# Sử dụng
pg = PostgresConnection(port=5432)
fs = s3fs.S3FileSystem()

data_dir = 's3://flezi-data/nyc-taxi/'
list_files = fs.listdir(data_dir)

with pg as conn:
    create_table(conn)
    for idx, item in enumerate(list_files):
        filename = item['Key'].split('/')[-1]
        if item['Key'].endswith('.parquet'):
            filepath = os.path.join(data_dir, filename)
            insert_parquet(conn, filepath=filepath, n=10)
            # break

Table 'nyc_taxi' created successfully.
Inserted 10/10 rows
Done! Inserted 10 rows from s3://flezi-agent/nyc-taxi/yellow_tripdata_2009-01.parquet
Inserted 10/10 rows
Done! Inserted 10 rows from s3://flezi-agent/nyc-taxi/yellow_tripdata_2009-02.parquet
Inserted 10/10 rows
Done! Inserted 10 rows from s3://flezi-agent/nyc-taxi/yellow_tripdata_2009-03.parquet
Inserted 10/10 rows
Done! Inserted 10 rows from s3://flezi-agent/nyc-taxi/yellow_tripdata_2009-04.parquet
Inserted 10/10 rows
Done! Inserted 10 rows from s3://flezi-agent/nyc-taxi/yellow_tripdata_2009-05.parquet
Inserted 10/10 rows
Done! Inserted 10 rows from s3://flezi-agent/nyc-taxi/yellow_tripdata_2009-06.parquet
Inserted 10/10 rows
Done! Inserted 10 rows from s3://flezi-agent/nyc-taxi/yellow_tripdata_2009-07.parquet
Inserted 10/10 rows
Done! Inserted 10 rows from s3://flezi-agent/nyc-taxi/yellow_tripdata_2009-08.parquet
Inserted 10/10 rows
Done! Inserted 10 rows from s3://flezi-agent/nyc-taxi/yellow_tripdata_2009-09.parquet
Inserte